In [33]:
!pip install -U pymoo

In [3]:
!pip install git+https://github.com/msu-coinlab/pymoo.git

  Cloning https://github.com/msu-coinlab/pymoo.git to /tmp/pip-req-build-it2i3ma2
  Running command git clone --filter=blob:none --quiet https://github.com/msu-coinlab/pymoo.git /tmp/pip-req-build-it2i3ma2
  Resolved https://github.com/msu-coinlab/pymoo.git to commit 78993eedad0d4d8c026d133071d1aab062bf4014
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.1/249.1 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 11.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pymoo: filename=pymoo-0.6.0.1-cp310-cp310-linux_x86_64.whl size=3613875 sha256=b7a13adef626ff528fcc88c53c6717c106e8e1878371a1b715596ff642bccd78
  Stored in directory: /tmp/pip-ephem-wheel-cache-zs7auo12/wheels/9f/88/ad/010bd246f26fa0906fb2964fc7629bfbfdeaf011e7

In [36]:
# # %git clone https://github.com/anyoptimization/pymoo
# # %cd pymoo
# # !pip install pymoo
# !pip install -qU git+https://github.com/anyoptimization/pymoo
# %cd pymoo


  Preparing metadata (setup.py) ... done


In [ ]:
pip install --upgrade pymoo

In [ ]:
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.problems import get_problem
from pymoo.optimize import minimize
from pymoo.visualization.scatter import Scatter
import numpy as np
from pymoo.core.problem import Problem
import matplotlib.pyplot as plt
from pymoo.gradient.toolbox import column_stack

In [46]:
from pymoo.problems.multi import OSY
%psource OSY

In [6]:
class ConesPB(Problem):
    def __init__(self):
        super().__init__(n_var = 2, n_obj = 2, n_ieq_constr = 1, xl= np.array([0, 0 ]), xu = np.array([10, 20]), vtype = float)


    def _evaluate(self, x, out, *args, **kwargs):
        r, h = x[:, 0], x[:, 1]
        s = np.sqrt(r ** 2 + h ** 2)

        B = np.pi * (r ** 2)
        S = np.pi * r * s
        T = B + S
        out["F"] = column_stack([S, T])

        V = (np.pi / 3) * (r ** 2) * h
        out["G"] = (-(1 / 200) * (V - 200))


    def calculate_pareto_front(self):
        algorithm = NSGA2(pop_size = 500)
        res = minimize(self, algorithm, termination = ('n_gen', 100), seed = 1)
        pareto_front_indices = self.find_pareto_front(res.F)

        return res.X[pareto_front_indices], res.F[pareto_front_indices]


    def find_pareto_front(self, objectives):
        n = len(objectives)
        is_pareto = np.ones(n, dtype=bool)
        for i in range(n):
            for j in range(n):
                if i != j and all(objectives[j] <= objectives[i]):
                    is_pareto[i] = 0
                    break
        return np.where(is_pareto)[0]

In [ ]:
cone_problem = ConesPB()
problem = cone_problem

algorithm = NSGA2(pop_size=300)

res = minimize(problem, algorithm, termination=('n_gen', 100), seed=1)



decision_variables = res.X
objective_values = res.F

# Extract Pareto front solutions
pareto_front = non_dominated(objective_values)


plot = Scatter()
plot.add(res.X, facecolor="none")
plot.show()

